# Bollinger Band Reversal (BBR)

## Import Libs

In [3]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Functions

### Get FE Data

In [4]:
def get_fe_price_data(
        filename: str = "FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    FOLDER = "price_data"
    PATH = f"{os.getcwd()}/{FOLDER}"
    df = pd.read_csv(f"{PATH}/{filename}")
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

In [5]:
# Set new columns 
gains_cols = [ 
    "Win", "Loss", 
    "TP", "SL", 
    "Gain",
    "Trade_Start", "Trade_End"
    ]

### Simulate Short Positions (Range-Based)

In [6]:
def range_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        bbl: Series,
        atr4: Series,
        sma4_slope: Series,
        range_type: str = "ADR",
        momentum_trade_mgmt = True
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        bbl_window = bbl.loc[START+TD:END]
        atr4_window = atr4.loc[START+TD:END]
        sma4_slope_window = sma4_slope.loc[START+TD:END]
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        momentum_stop = False

        for i in range(len(sl_window)): 
            stop_condition = sl_window.iloc[i] >= (df["Close"] + (df[range_type] * sl_pct_range))
            momentum_condition = close_window.iloc[i] < bbl_window.iloc[i] if momentum_trade_mgmt is True else False
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
            if momentum_condition:
                momentum_stop = True
                sl_ts = close_window.iloc[i:i+1].index[0]
                break

        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False and momentum_stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = df["Close"] - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["Close"] - tp
        sl_pips = df["Close"] - (df["Close"] + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            if trade_start == trade_end:
                if stop is False:
                    win = 1
                    gain = df["Close"] - tp
                elif momentum_stop is True:
                    gain = df["Close"] - tp
                    if gain > 0:
                        win = 1
                    else:
                        loss = 1
                else:
                    loss = 1
                    gain = sl_pips
            elif momentum_stop is True:
                gain = df["Close"] - max(tp, close.loc[sl_ts])
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                win = 1
                gain = df["Close"] - tp                                
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            elif momentum_stop is True:
                gain = df["Close"] - close.loc[sl_ts]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data


### Simulate Long Positions (Range-Based)

In [7]:
def range_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        bbu: Series,
        atr4: Series,
        sma4_slope: Series,
        range_type: str = "ADR",
        momentum_trade_mgmt = True
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        bbu_window = bbu.loc[START+TD:END]
        atr4_window = atr4.loc[START+TD:END]
        sma4_slope_window = sma4_slope.loc[START+TD:END]
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        momentum_stop = False

        for i in range(len(sl_window)):
            stop_condition = sl_window.iloc[i] <= (df["Close"] - (df[range_type] * sl_pct_range))
            momentum_condition = close_window.iloc[i] > bbu_window.iloc[i] if momentum_trade_mgmt is True else False
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
            if momentum_condition:
                momentum_stop = True
                sl_ts = close_window.iloc[i:i+1].index[0]
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False and momentum_stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - df["Close"]
        sl_pips = (df["Close"] - df[range_type] * sl_pct_range) - df["Close"] 
        if tp_window.max() >= tp:
            if trade_start == trade_end:
                if stop is False:
                    win = 1
                    gain = tp - df["Close"]
                elif momentum_stop is True:
                    gain = tp - df["Close"]
                    if gain > 0:
                        win = 1
                    else:
                        loss = 1
                else:
                    loss = 1
                    gain = sl_pips
            elif momentum_stop is True:
                gain = min(tp, close.loc[sl_ts]) - df["Close"]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                win = 1
                gain = tp - df["Close"]
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            elif momentum_stop is True:
                gain = close.loc[sl_ts] - df["Close"]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data


### Simulate Limit Short Position

### Simulate Limit Long Position

### Limit-Based Gains (Long/Short)

### Range-Based Gains (Long/Short)

In [8]:
def get_range_signal_gains(df: DataFrame, long: str, short: str, sl_pct_r, tp_pct_r):
    long_df = df.copy() 
    short_df = df.copy()
    # long
    long_df[gains_cols] = long_df.apply(
        range_long_gains,
        axis=1,
        args=[long_df["High"],long_df["Low"],long_df["Close"],
            long, sl_pct_r, tp_pct_r, long_df["BB_Upper_16_2"], long_df["ATR4"], long_df["SMA4_Slope"]],
        result_type='expand',
        range_type="ATR4",
        momentum_trade_mgmt = True
    )
    # short
    short_df[gains_cols] = short_df.apply(
        range_short_gains,
        axis=1,
        args=[short_df["High"],short_df["Low"],short_df["Close"],
            short, sl_pct_r, tp_pct_r, short_df["BB_Lower_16_2"], short_df["ATR4"], short_df["SMA4_Slope"]],
        result_type='expand',
        range_type="ATR4",
        momentum_trade_mgmt = True
    )
    return pd.concat([long_df, short_df])

### Simulate Long/Short Trend Position

In [9]:
def trend_gains(
        df: Series, 
        close: Series,
        sma_fast: Series,
        sma_slow: Series,
        long_signal: str,
        short_signal: str
        ):
    """Get the pip gain and apply to df
    
    - long_signal: name of buy signal
    - short_signal: name of sell signal
    """

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None

    if df[long_signal] is True or df[short_signal] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = close.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        sma_fast_window = sma_fast.loc[START+TD:END] # from signal idx+1 to EOD
        sma_slow_window = sma_slow.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        
        # find exit timestamp:
        for i in range(len(close_window)): 
            # exit condition
            if df[long_signal] is True:
                exit_condition = 1 if close_window.iloc[i] < sma_fast_window.iloc[i] else 0
            elif df[short_signal] is True:
                exit_condition = 1 if close_window.iloc[i] > sma_fast_window.iloc[i] else 0
            # get stop loss time:
            if exit_condition == 1:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = close_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD if (START+TD) < END else START

        # Win / Loss / Gain / Pips
        if df[long_signal] is True:
            gain = close[sl_ts] - df["Close"]
        elif df[short_signal] is True:
            gain = df["Close"] - close[sl_ts]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["ATR4"]
        sl_pips = -(df["ATR4"])
        win = 1 if gain > 0 else 0
        loss = 1 if gain < 0 else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data



### Trade Stats

In [10]:
def trade_stats(df: DataFrame, signal_name: str, trade_size: int = 500000, comm: float = 0.00002, quote_ccy: float = 1.34):
    win_count = df.query(f"{signal_name} == True and Win > 0")[f"{signal_name}"].count()
    loss_count = df.query(f"{signal_name} == True and Loss > 0")[f"{signal_name}"].count()
    total_trades = win_count + loss_count
    win_rate = win_count/total_trades * 100
    win = df.query(f"{signal_name} == True and Gain > 0")["Gain"]
    loss = df.query(f"{signal_name} == True and Gain < 0")["Gain"]
    win_avg_pips = win.mean()
    loss_avg_pips = loss.mean()
    win_pips = win.sum()
    loss_pips = loss.sum()
    total_pips = win_pips + loss_pips
    trade_value = trade_size * quote_ccy
    comm = comm * 2 * trade_value

    stats = {
        "Symbol": df["Symbol"].iloc[0],
        "Start": df.index.min(),
        "End": df.index.max(),
        "Win_Count": win_count,
        "Loss_Count": loss_count,
        "Total_Trades": total_trades,
        "Win_Rate": win_rate,
        "Avg_Win": win_avg_pips,
        "Avg_Loss": loss_avg_pips,
        "Win_Pips": win_pips,
        "Loss_Pips": loss_pips,
        "Total_Pips": total_pips,
        "Trade_Value": trade_value,
        "Commissions": comm,
        "Return ($)": round((total_pips * trade_value) - (comm * total_trades),2)
    }

    return stats

def signal_stats(df: DataFrame, signals: list["str"]):
    stats = [trade_stats(df, signal) for signal in signals]
    return stats

def get_trend_signal_stats(df: DataFrame, long: str, short: str, sma_fast: str = "SMA8", sma_slow: str = "SMA16"):
    # Apply gains to df
    df[gains_cols] = df.apply(
        trend_gains,
        axis=1,
        args=[df["Close"], df[sma_fast], df[sma_slow], long, short],
        result_type='expand'
    )

    # get stats for long / short signals
    gain_stats = signal_stats(df, [long,short])

    # build stats dataframe
    return pd.DataFrame(data=gain_stats,
                index=[long,short]
                )

## Price Data Files 

In [72]:
price_data_files = os.listdir(f"{os.getcwd()}/price_data")
# section filenames by currency pairs
gbp_data = [x for x in price_data_files if "GBPUSD" in x]
gbp_data.sort()
eur_data = [x for x in price_data_files if "EUR" in x]
eur_data.sort()
cad_data = [x for x in price_data_files if "CAD" in x]
cad_data.sort()
jpy_data = [x for x in price_data_files if "JPY" in x]
jpy_data.sort()
aud_data = [x for x in price_data_files if "AUDUSD" in x]
aud_data.sort()
nzd_data = [x for x in price_data_files if "NZD" in x]
nzd_data.sort()
chf_data = [x for x in price_data_files if "CHF" in x]
chf_data.sort()
gbp_aud_data = [x for x in price_data_files if "GBPAUD" in x]
gbp_aud_data.sort()
latest = [x for x in price_data_files if "latest" in x]
# Set max row output to 100 rows
pd.options.display.max_rows = 100

### Feature-Enriched Dataframe Lists

In [69]:
# get dataframe of feature enriched price data for each year (file)
gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]
eur_df_list = [get_fe_price_data(filename=x) for x in eur_data]
cad_df_list = [get_fe_price_data(filename=x) for x in cad_data]
jpy_df_list = [get_fe_price_data(filename=x) for x in jpy_data]
aud_df_list = [get_fe_price_data(filename=x) for x in aud_data]
nzd_df_list = [get_fe_price_data(filename=x) for x in nzd_data]
chf_df_list = [get_fe_price_data(filename=x) for x in chf_data]
latest_df_list = [get_fe_price_data(filename=x) for x in latest]
gbp_aud_df_list = [get_fe_price_data(filename=x) for x in gbp_aud_data]

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (65,66,73,77,78,83) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (63,65,68,73,74,76,78,82,83,98) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (47,61,63,65,66,67,68,73,74,75,76,77,78,80,81,82,83,91,92,98) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (61,62,63,64,65,66,67,68,73,74,75,76,77,78,80,81,82,83,91,92,97,98) have mixed types. Specify dtype option on import or s

# Multi-Year Results

## Multi-Year Functions

In [90]:
def range_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str, 
        sl_pct_r: float, 
        tp_pct_r: float,
        get_signal_gains_func: function,
        trading_session: str = ""
        ):
    session_start = ""
    session_end = ""
    if trading_session == "london":
        session_start = "02:00"
        session_end = "11:00"
    elif trading_session == "new york":
        session_start = "07:00"
        session_end = "16:00"
    elif trading_session == "asia":
        session_start = "19:00"
        session_end = "05:00"
    else:
        session_start = "17:15"
        session_end = "16:45"
    signal_stats_list = []
    gains_df_list = []
    for df in df_list:
        signal_gains_df: DataFrame = get_signal_gains_func(df, long, short, sl_pct_r, tp_pct_r)
        gains_df_list.append(signal_gains_df)
        long_short_stats = signal_stats(signal_gains_df.between_time(session_start,session_end), [long,short])
        stats_df = pd.DataFrame(data=long_short_stats, index=[long,short])
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list), pd.concat(gains_df_list)

def trend_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str,
        sma: str
        ):
    signal_stats_list = []
    for df in df_list:
        stats_df = get_trend_signal_stats(df, long, short, sma)
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list)

## GBP/USD

In [177]:
gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (67,68,75,79,80,85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (65,67,70,75,76,78,80,84,85,98) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (47,63,65,67,68,69,70,75,76,77,78,79,80,82,83,84,85,91,92,98) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_18875/1547086338.py:10: DtypeWarning: Columns (63,64,65,66,67,68,69,70,75,76,77,78,79,80,82,83,84,85,91,92,97,98) have mixed types. Specify dtype option on import or s

### Extreme Momentum

In [14]:
long_signal = "Bull_XM_V2"
short_signal = "Bear_XM_V2"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 1, 1, get_range_signal_gains, trading_session="london")

#### Results (5 Years)

In [15]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats
# stats["Total_Pips"].sum()
# stats["Win_Count"].mean(), stats["Loss_Count"].mean()
# stats["Win_Rate"].mean()
# stats["Total_Pips"].mean()
# stats["Return ($)"].mean() * 2

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_XM_V2,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,173,143,316,54.746835,0.001021,-0.001139,0.176652,-0.162879,0.013774,670000.0,26.8,759.61,759.61
Bear_XM_V2,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,195,132,327,59.633028,0.001272,-0.001317,0.247967,-0.173821,0.074146,670000.0,26.8,40914.39,41674.00
Bull_XM_V2,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,169,148,317,53.312303,0.001856,-0.001979,0.313742,-0.292890,0.020853,670000.0,26.8,5475.58,47149.58
Bear_XM_V2,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,251,198,449,55.902004,0.001768,-0.001847,0.443796,-0.365795,0.078001,670000.0,26.8,40227.64,87377.22
Bull_XM_V2,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,195,172,367,53.133515,0.001104,-0.001132,0.215226,-0.190137,0.025089,670000.0,26.8,6973.86,94351.08
Bear_XM_V2,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,170,148,318,53.459119,0.001230,-0.001353,0.209039,-0.192080,0.016959,670000.0,26.8,2839.96,97191.04
Bull_XM_V2,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,158,140,298,53.020134,0.000911,-0.001188,0.143887,-0.166270,-0.022382,670000.0,26.8,-22982.67,74208.37
Bear_XM_V2,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,133,100,233,57.081545,0.001235,-0.001224,0.164220,-0.122410,0.041810,670000.0,26.8,21768.30,95976.67
Bull_XM_V2,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,186,153,339,54.867257,0.001164,-0.001120,0.216530,-0.171436,0.045094,670000.0,26.8,21127.61,117104.28
Bear_XM_V2,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,171,132,303,56.435644,0.001251,-0.001247,0.213975,-0.164637,0.049337,670000.0,26.8,24935.72,142040.00


#### Trades (Last 10)

In [16]:
g.loc["2026"][[*gains_cols, "SMA4_Slope", "Bull_XM_V2", "Bear_XM_V2"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,Bull_XM_V2,Bear_XM_V2
Date,,,,,,,,,,
2026-03-03 06:00:00-05:00,0.0,1.0,0.001799,-0.001799,-0.001799,2026-03-03 06:15:00-05:00,2026-03-03 06:45:00-05:00,-71.916555,False,True
2026-03-03 08:30:00-05:00,1.0,0.0,0.001389,-0.001389,0.001389,2026-03-03 08:45:00-05:00,2026-03-03 11:15:00-05:00,-61.714568,False,True
2026-03-03 08:45:00-05:00,1.0,0.0,0.001317,-0.001317,0.001317,2026-03-03 09:00:00-05:00,2026-03-03 11:15:00-05:00,-72.299572,False,True
2026-03-03 09:00:00-05:00,0.0,1.0,0.001197,-0.001197,-0.001197,2026-03-03 09:15:00-05:00,2026-03-03 09:15:00-05:00,-69.619111,False,True
2026-03-03 09:15:00-05:00,1.0,0.0,0.001494,-0.001494,0.001494,2026-03-03 09:30:00-05:00,2026-03-03 11:15:00-05:00,-59.224785,False,True
2026-03-03 09:30:00-05:00,1.0,0.0,0.001762,-0.001762,0.001762,2026-03-03 09:45:00-05:00,2026-03-03 10:30:00-05:00,-63.950691,False,True
2026-03-03 09:45:00-05:00,1.0,0.0,0.001885,-0.001885,0.001885,2026-03-03 10:00:00-05:00,2026-03-03 10:30:00-05:00,-69.819881,False,True
2026-03-03 10:00:00-05:00,0.0,1.0,0.002221,-0.002221,-0.002221,2026-03-03 10:15:00-05:00,2026-03-03 10:30:00-05:00,-74.628325,False,True
2026-03-11 09:00:00-04:00,0.0,1.0,0.001691,-0.001691,-0.001691,2026-03-11 09:15:00-04:00,2026-03-11 10:00:00-04:00,-74.492972,False,True


### Trend Continuation

In [17]:
long_signal = "Bull_TC"
short_signal = "Bear_TC"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 1, 1.5, get_range_signal_gains, trading_session="london")

#### Results (5 Years)

In [18]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_TC,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,113,152,265,42.641509,0.001550,-0.001113,0.175169,-0.169135,0.006034,670000.0,26.8,-3059.39,-3059.39
Bear_TC,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,121,132,253,47.826087,0.001552,-0.001214,0.187819,-0.160199,0.027621,670000.0,26.8,11725.42,8666.03
Bull_TC,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,129,159,288,44.791667,0.002321,-0.001658,0.299468,-0.263557,0.035911,670000.0,26.8,16341.72,25007.75
Bear_TC,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,147,152,299,49.163880,0.002210,-0.001793,0.324811,-0.272465,0.052346,670000.0,26.8,27058.37,52066.12
Bull_TC,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,115,120,235,48.936170,0.001525,-0.001200,0.175429,-0.144031,0.031398,670000.0,26.8,14738.74,66804.86
Bear_TC,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,104,125,229,45.414847,0.001452,-0.001257,0.150984,-0.155919,-0.004935,670000.0,26.8,-9443.65,57361.21
Bull_TC,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,115,114,229,50.218341,0.001200,-0.000967,0.137997,-0.110284,0.027714,670000.0,26.8,12431.01,69792.22
Bear_TC,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,78,93,171,45.614035,0.001239,-0.001115,0.096628,-0.103726,-0.007098,670000.0,26.8,-9338.54,60453.68
Bull_TC,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,111,127,238,46.638655,0.001568,-0.001144,0.174011,-0.145341,0.028669,670000.0,26.8,12830.08,73283.76
Bear_TC,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,124,116,240,51.666667,0.001412,-0.001084,0.175091,-0.125694,0.049397,670000.0,26.8,26663.91,99947.67


#### Trades (Last 10)

In [19]:
g[[*gains_cols, "SMA16_Slope", "SMA32_Slope", "Bull_TC", "Bear_TC"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA16_Slope,SMA32_Slope,Bull_TC,Bear_TC
Date,,,,,,,,,,,
2026-03-03 08:00:00-05:00,1.0,0.0,0.002402,-0.001601,0.002402,2026-03-03 08:15:00-05:00,2026-03-03 11:45:00-05:00,22.518081,-53.941633,False,True
2026-03-03 09:30:00-05:00,1.0,0.0,0.002644,-0.001762,0.002644,2026-03-03 09:45:00-05:00,2026-03-03 10:30:00-05:00,-14.819918,-55.204685,False,True
2026-03-04 10:45:00-05:00,1.0,0.0,0.002430,-0.001620,0.001505,2026-03-04 11:00:00-05:00,2026-03-04 12:15:00-05:00,-22.058201,6.359843,False,True
2026-03-05 02:45:00-05:00,1.0,0.0,0.002076,-0.001384,0.002076,2026-03-05 03:00:00-05:00,2026-03-05 04:00:00-05:00,12.053992,-25.456395,False,True
2026-03-05 03:00:00-05:00,0.0,1.0,0.002539,-0.001692,-0.001692,2026-03-05 03:15:00-05:00,2026-03-05 04:00:00-05:00,-18.863588,-37.812953,False,True
2026-03-05 05:30:00-05:00,0.0,1.0,0.001931,-0.001288,-0.001288,2026-03-05 05:45:00-05:00,2026-03-05 05:45:00-05:00,43.056947,-23.151754,False,True
2026-03-06 07:45:00-05:00,0.0,1.0,0.002151,-0.001434,-0.001434,2026-03-06 08:00:00-05:00,2026-03-06 08:00:00-05:00,-57.876877,-36.078796,False,True
2026-03-10 09:45:00-04:00,0.0,1.0,0.002182,-0.001455,-0.001455,2026-03-10 10:00:00-04:00,2026-03-10 10:30:00-04:00,-22.467133,24.178093,False,True
2026-03-11 10:30:00-04:00,1.0,0.0,0.002799,-0.001866,0.002799,2026-03-11 10:45:00-04:00,2026-03-11 16:45:00-04:00,-31.095229,-28.142141,False,True


### Trend Momentum

In [183]:
long_signal = "Bull_Pullback"
short_signal = "Bear_Pullback"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 1, 1, get_range_signal_gains, trading_session="london")

#### Results (5 Years)

In [185]:
g: DataFrame = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_Pullback,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,680,587,1267,53.670087,0.001051,-0.001113,0.714674,-0.653426,0.061247,670000.0,26.8,7080.22,7080.22
Bear_Pullback,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,722,596,1318,54.779970,0.001182,-0.001233,0.853715,-0.735112,0.118603,670000.0,26.8,44141.28,51221.50
Bull_Pullback,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,755,707,1462,51.641587,0.001762,-0.001837,1.330132,-1.298909,0.031224,670000.0,26.8,-18261.69,32959.81
Bear_Pullback,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,899,883,1782,50.448934,0.001687,-0.001753,1.516817,-1.547771,-0.030954,670000.0,26.8,-68496.61,-35536.80
Bull_Pullback,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,640,578,1218,52.545156,0.001104,-0.001155,0.706747,-0.667839,0.038909,670000.0,26.8,-6573.54,-42110.34
Bear_Pullback,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,683,638,1321,51.703255,0.001149,-0.001166,0.784699,-0.731169,0.053530,670000.0,26.8,462.30,-41648.04
Bull_Pullback,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,544,628,1172,46.416382,0.000940,-0.001066,0.511585,-0.669647,-0.158062,670000.0,26.8,-137311.47,-178959.51
Bear_Pullback,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,559,475,1034,54.061896,0.001080,-0.001156,0.603971,-0.549191,0.054780,670000.0,26.8,8991.40,-169968.11
Bull_Pullback,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,691,619,1310,52.748092,0.001114,-0.001102,0.769464,-0.682012,0.087451,670000.0,26.8,23484.34,-146483.77
Bear_Pullback,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,525,559,1084,48.431734,0.001171,-0.001179,0.614537,-0.657666,-0.043129,670000.0,26.8,-57947.46,-204431.23


#### Trades (Last 10)

In [218]:
t = g.loc["2022-03-11 02:00:00-05:00":][[*gains_cols, "SMA4_Slope","SMA16_Slope", "SMA32_Slope","SMA16_Slope_SMA","SMA32_Slope_SMA", "Bull_Pullback", "Bear_Pullback"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00")
t.sort_index().iloc[0:50]
# t.iloc[150:200]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,SMA16_Slope,SMA32_Slope,SMA16_Slope_SMA,SMA32_Slope_SMA,Bull_Pullback,Bear_Pullback
Date,,,,,,,,,,,,,,
2022-03-14 08:30:00-04:00,0.0,1.0,0.001630,-0.001630,-0.001630,2022-03-14 08:45:00-04:00,2022-03-14 09:15:00-04:00,80.702539,38.072782,44.745221,28.295642,20.592709,True,NaN
2022-03-14 08:45:00-04:00,0.0,1.0,0.001829,-0.001829,-0.001829,2022-03-14 09:00:00-04:00,2022-03-14 09:45:00-04:00,82.667117,47.454846,46.453895,28.423148,23.341370,True,NaN
2022-03-14 09:00:00-04:00,0.0,1.0,0.001529,-0.001529,-0.001529,2022-03-14 09:15:00-04:00,2022-03-14 09:45:00-04:00,78.407825,35.981213,41.320858,27.802579,25.840010,True,NaN
2022-03-14 09:15:00-04:00,0.0,1.0,0.001507,-0.001507,-0.001507,2022-03-14 09:30:00-04:00,2022-03-14 09:45:00-04:00,63.339137,22.772258,41.016474,25.922785,28.006049,True,NaN
2022-03-15 02:00:00-04:00,0.0,1.0,0.000962,-0.000962,-0.000962,2022-03-15 02:15:00-04:00,2022-03-15 02:45:00-04:00,27.512003,54.055052,38.732519,47.466275,27.260879,True,NaN
2022-03-15 02:15:00-04:00,0.0,1.0,0.000876,-0.000876,-0.000876,2022-03-15 02:30:00-04:00,2022-03-15 02:45:00-04:00,-33.855026,44.059815,35.765677,48.638123,30.078552,True,NaN
2022-03-15 02:30:00-04:00,0.0,1.0,0.000785,-0.000785,-0.000785,2022-03-15 02:45:00-04:00,2022-03-15 02:45:00-04:00,-46.623333,31.962445,33.023868,48.507005,32.013907,True,NaN
2022-03-15 09:15:00-04:00,0.0,1.0,0.000974,-0.000974,-0.000974,2022-03-15 09:30:00-04:00,2022-03-15 09:30:00-04:00,50.869600,65.657843,33.648729,27.240103,24.101752,True,NaN
2022-03-15 09:30:00-04:00,0.0,1.0,0.001065,-0.001065,-0.001065,2022-03-15 09:45:00-04:00,2022-03-15 11:15:00-04:00,24.227745,58.909390,31.574191,32.373652,24.214879,True,NaN


Note:
- momentum
    - SMA4_Slope and SMA4_Slope_SMA to control Momentum - (Trend reversal when sma16 < sma32 and sma4 > sma32 + sma4_slope > 45 and sma4_slope_sma > 45)
    - when there is momentum, you can buy / sell indiviual inside bars. For bullish momentum buy red inside bar, and for bearish sell green inside bar.
    - momentum reversal: if momentum is not longer true, and there is no uptrend, and mb_high is < 0.10 pct_dhigh and current bar is an inside bar, then sell at mb high

- trend
    - SMA16 under SMA32 for 4hrs/16bars (downtrend), and SMA16_Slope_SMA < 0 and SMA32_Slope < -11.25 to control trend
    - for inside bars in trend, if in uptrend buy the low of the mother bar, if in downtrend sell the high of mother bar.

In [283]:
x = g[[*gains_cols, "SMA4_Slope","SMA16_Slope", "SMA32_Slope","SMA4_Slope_SMA","SMA16_Slope_SMA","SMA32_Slope_SMA", "IB"]].query("'2022-07-21'")
x.iloc[0:100]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,SMA16_Slope,SMA32_Slope,SMA4_Slope_SMA,SMA16_Slope_SMA,SMA32_Slope_SMA,IB
Date,,,,,,,,,,,,,,
2022-07-18 00:00:00-04:00,0.0,1.0,0.000647,-0.000647,-0.000647,2022-07-18 00:15:00-04:00,2022-07-18 00:15:00-04:00,-28.627183,22.924312,35.251555,16.727345,43.158146,38.184603,False
2022-07-18 00:15:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,-64.403626,17.245214,26.779507,-7.960086,42.051475,38.026297,False
2022-07-18 00:30:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,-71.396464,9.636444,23.101278,-35.401073,40.507122,37.423233,False
2022-07-18 00:45:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,-68.618806,10.215535,21.362746,-58.261520,39.204877,36.637325,True
2022-07-18 01:00:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,-56.959762,-5.237476,23.403564,-65.344664,36.143160,35.740233,True
2022-07-18 01:15:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,2.385944,-13.867603,25.796026,-48.647272,32.119541,35.026727,False
2022-07-18 01:30:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,65.556045,1.372449,32.176735,-14.409145,29.018560,34.695512,False
2022-07-18 01:45:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,65.474030,-2.266775,33.482975,19.114064,25.519175,34.342586,True
2022-07-18 02:00:00-04:00,0.0,0.0,0.000000,0.000000,0.000000,NaT,NaT,24.227745,-24.028906,25.016893,39.410941,20.638928,33.433759,False


## AUD/USD

### Extreme Momentum

In [23]:
long_signal = "Bull_XM_V2"
short_signal = "Bear_XM_V2"

stats, gains = range_based_multi_year_stats(aud_df_list, long_signal, short_signal, 1, 1, get_range_signal_gains, trading_session="asia")

#### Results (5 Years)

In [24]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_XM_V2,AUDUSD,2021-03-11 19:00:00-05:00,2022-03-11 05:00:00-05:00,83,78,161,51.552795,0.000553,-0.000680,0.045861,-0.053029,-0.007167,670000.0,26.8,-9117.02,-9117.02
Bear_XM_V2,AUDUSD,2021-03-11 19:00:00-05:00,2022-03-11 05:00:00-05:00,95,71,166,57.228916,0.000712,-0.000786,0.067621,-0.055777,0.011844,670000.0,26.8,3486.51,-5630.51
Bull_XM_V2,AUDUSD,2022-03-10 19:00:00-05:00,2023-03-10 05:00:00-05:00,94,102,196,47.959184,0.000873,-0.000977,0.082066,-0.099630,-0.017564,670000.0,26.8,-17020.51,-22651.02
Bear_XM_V2,AUDUSD,2022-03-10 19:00:00-05:00,2023-03-10 05:00:00-05:00,145,131,276,52.536232,0.000933,-0.000976,0.135221,-0.126850,0.008371,670000.0,26.8,-1788.06,-24439.08
Bull_XM_V2,AUDUSD,2023-03-13 19:00:00-04:00,2024-03-12 05:00:00-04:00,83,92,175,47.428571,0.000694,-0.000655,0.057566,-0.059621,-0.002055,670000.0,26.8,-6066.85,-30505.93
Bear_XM_V2,AUDUSD,2023-03-13 19:00:00-04:00,2024-03-12 05:00:00-04:00,90,62,152,59.210526,0.000718,-0.000780,0.064604,-0.046010,0.018594,670000.0,26.8,8384.21,-22121.72
Bull_XM_V2,AUDUSD,2024-03-12 19:00:00-04:00,2025-03-12 05:00:00-04:00,52,64,116,44.827586,0.000617,-0.000620,0.032104,-0.039701,-0.007597,670000.0,26.8,-8199.12,-30320.84
Bear_XM_V2,AUDUSD,2024-03-12 19:00:00-04:00,2025-03-12 05:00:00-04:00,81,68,149,54.362416,0.000707,-0.000878,0.057297,-0.059737,-0.002440,670000.0,26.8,-5628.00,-35948.84
Bull_XM_V2,AUDUSD,2025-03-12 19:00:00-04:00,2026-03-12 05:00:00-04:00,88,78,166,53.012048,0.000574,-0.000646,0.050525,-0.050414,0.000111,670000.0,26.8,-4374.26,-40323.10
Bear_XM_V2,AUDUSD,2025-03-12 19:00:00-04:00,2026-03-12 05:00:00-04:00,70,47,117,59.829060,0.000729,-0.000866,0.051039,-0.040712,0.010326,670000.0,26.8,3782.99,-36540.11


#### Trades (Last 10)

In [25]:
g[[*gains_cols, "SMA4_Slope", "Bull_XM_V2", "Bear_XM_V2"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,Bull_XM_V2,Bear_XM_V2
Date,,,,,,,,,,
2026-03-03 05:45:00-05:00,1.0,0.0,0.001394,-0.001394,0.001394,2026-03-03 06:00:00-05:00,2026-03-03 07:15:00-05:00,-67.965750,False,True
2026-03-03 06:00:00-05:00,0.0,1.0,0.001430,-0.001430,-0.001430,2026-03-03 06:15:00-05:00,2026-03-03 07:00:00-05:00,-72.431031,False,True
2026-03-03 06:15:00-05:00,0.0,1.0,0.001607,-0.001607,-0.001607,2026-03-03 06:30:00-05:00,2026-03-03 07:00:00-05:00,-73.060914,False,True
2026-03-03 06:30:00-05:00,0.0,1.0,0.001341,-0.001341,-0.001341,2026-03-03 06:45:00-05:00,2026-03-03 07:00:00-05:00,-67.130342,False,True
2026-03-03 09:00:00-05:00,1.0,0.0,0.001161,-0.001161,0.001161,2026-03-03 09:15:00-05:00,2026-03-03 09:30:00-05:00,-71.125450,False,True
2026-03-03 09:15:00-05:00,1.0,0.0,0.001165,-0.001165,0.001165,2026-03-03 09:30:00-05:00,2026-03-03 09:30:00-05:00,-68.066115,False,True
2026-03-03 09:45:00-05:00,1.0,0.0,0.001671,-0.001671,0.001671,2026-03-03 10:00:00-05:00,2026-03-03 10:00:00-05:00,-79.020016,False,True
2026-03-03 10:15:00-05:00,0.0,1.0,0.002804,-0.002804,-0.002804,2026-03-03 10:30:00-05:00,2026-03-03 11:00:00-05:00,-81.613851,False,True
2026-03-12 10:15:00-04:00,1.0,0.0,0.001326,-0.001326,0.001326,2026-03-12 10:30:00-04:00,2026-03-12 16:45:00-04:00,-74.544001,False,True


### Trend Continuation

In [26]:
long_signal = "Bull_TC"
short_signal = "Bear_TC"

stats, gains = range_based_multi_year_stats(aud_df_list, long_signal, short_signal, 1, 1.5, get_range_signal_gains, trading_session="asia")

#### Results (5 Years)

In [27]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_TC,AUDUSD,2021-03-11 19:00:00-05:00,2022-03-11 05:00:00-05:00,72,101,173,41.618497,0.000851,-0.000640,0.061282,-0.064666,-0.003384,670000.0,26.8,-6903.93,-6903.93
Bear_TC,AUDUSD,2021-03-11 19:00:00-05:00,2022-03-11 05:00:00-05:00,69,112,181,38.121547,0.000802,-0.000636,0.055360,-0.071194,-0.015834,670000.0,26.8,-15459.41,-22363.34
Bull_TC,AUDUSD,2022-03-10 19:00:00-05:00,2023-03-10 05:00:00-05:00,103,122,225,45.777778,0.001116,-0.000906,0.114962,-0.110555,0.004408,670000.0,26.8,-3076.97,-25440.31
Bear_TC,AUDUSD,2022-03-10 19:00:00-05:00,2023-03-10 05:00:00-05:00,130,99,229,56.768559,0.001030,-0.000810,0.133924,-0.080210,0.053714,670000.0,26.8,29851.43,4411.12
Bull_TC,AUDUSD,2023-03-13 19:00:00-04:00,2024-03-12 05:00:00-04:00,66,96,162,40.740741,0.000753,-0.000601,0.049669,-0.056464,-0.006794,670000.0,26.8,-8893.83,-4482.71
Bear_TC,AUDUSD,2023-03-13 19:00:00-04:00,2024-03-12 05:00:00-04:00,67,67,134,50.000000,0.000782,-0.000659,0.052368,-0.043471,0.008897,670000.0,26.8,2369.71,-2113.00
Bull_TC,AUDUSD,2024-03-12 19:00:00-04:00,2025-03-12 05:00:00-04:00,51,81,132,38.636364,0.000713,-0.000492,0.036362,-0.039832,-0.003470,670000.0,26.8,-5862.50,-7975.50
Bear_TC,AUDUSD,2024-03-12 19:00:00-04:00,2025-03-12 05:00:00-04:00,71,97,168,42.261905,0.000688,-0.000563,0.048864,-0.054646,-0.005782,670000.0,26.8,-8376.26,-16351.76
Bull_TC,AUDUSD,2025-03-12 19:00:00-04:00,2026-03-12 05:00:00-04:00,83,86,169,49.112426,0.000862,-0.000664,0.071557,-0.057075,0.014483,670000.0,26.8,5174.08,-11177.68
Bear_TC,AUDUSD,2025-03-12 19:00:00-04:00,2026-03-12 05:00:00-04:00,59,77,136,43.382353,0.000789,-0.000646,0.046568,-0.049705,-0.003137,670000.0,26.8,-5746.51,-16924.19


#### Trades (Last 10)

In [28]:
g[[*gains_cols, "SMA16_Slope", "SMA32_Slope", "Bull_TC", "Bear_TC"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA16_Slope,SMA32_Slope,Bull_TC,Bear_TC
Date,,,,,,,,,,,
2026-03-03 08:00:00-05:00,1.0,0.0,0.002164,-0.001442,0.002164,2026-03-03 08:15:00-05:00,2026-03-03 09:30:00-05:00,-53.471145,-58.628612,False,True
2026-03-05 05:30:00-05:00,0.0,1.0,0.001101,-0.000734,-0.000734,2026-03-05 05:45:00-05:00,2026-03-05 06:00:00-05:00,21.129440,-25.237041,False,True
2026-03-05 08:15:00-05:00,1.0,0.0,0.001566,-0.001044,0.001566,2026-03-05 08:30:00-05:00,2026-03-05 08:30:00-05:00,-30.434236,-9.694444,False,True
2026-03-06 07:45:00-05:00,0.0,1.0,0.001399,-0.000933,-0.000933,2026-03-06 08:00:00-05:00,2026-03-06 08:00:00-05:00,-55.561011,-31.051445,False,True
2026-03-06 10:15:00-05:00,0.0,1.0,0.002977,-0.001985,-0.001985,2026-03-06 10:30:00-05:00,2026-03-06 13:15:00-05:00,1.611019,-28.373667,False,True
2026-03-06 10:45:00-05:00,0.0,1.0,0.002878,-0.001919,-0.001919,2026-03-06 11:00:00-05:00,2026-03-06 11:30:00-05:00,29.809114,-36.850794,False,True
2026-03-11 04:15:00-04:00,1.0,0.0,0.001779,-0.001186,0.000930,2026-03-11 04:30:00-04:00,2026-03-11 04:30:00-04:00,-21.595310,38.586950,False,True
2026-03-11 08:30:00-04:00,0.0,1.0,0.001785,-0.001190,-0.001190,2026-03-11 08:45:00-04:00,2026-03-11 09:45:00-04:00,-2.921936,-21.492040,False,True
2026-03-12 05:00:00-04:00,0.0,1.0,0.001136,-0.000757,-0.000757,2026-03-12 05:15:00-04:00,2026-03-12 05:30:00-04:00,15.098449,-16.644473,False,True


## USD/CAD

### Extreme Momentum

In [29]:
long_signal = "Bull_XM_V2"
short_signal = "Bear_XM_V2"

stats, gains = range_based_multi_year_stats(cad_df_list, long_signal, short_signal, 1, 1, get_range_signal_gains, trading_session="london")

#### Results (5 Years)

In [30]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_XM_V2,USDCAD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,192,147,339,56.637168,0.001082,-0.001221,0.207802,-0.179449,0.028354,670000.0,26.8,9911.81,9911.81
Bear_XM_V2,USDCAD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,112,106,218,51.376147,0.000998,-0.001094,0.111794,-0.115912,-0.004119,670000.0,26.8,-8601.96,1309.85
Bull_XM_V2,USDCAD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,227,192,419,54.176611,0.001219,-0.001348,0.276655,-0.258835,0.017820,670000.0,26.8,710.20,2020.05
Bear_XM_V2,USDCAD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,113,135,248,45.564516,0.001159,-0.001315,0.130985,-0.177561,-0.046576,670000.0,26.8,-37852.49,-35832.44
Bull_XM_V2,USDCAD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,151,179,330,45.757576,0.000949,-0.001011,0.143262,-0.179002,-0.035740,670000.0,26.8,-32789.80,-68622.24
Bear_XM_V2,USDCAD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,161,122,283,56.890459,0.000900,-0.000999,0.144871,-0.119914,0.024958,670000.0,26.8,9137.13,-59485.11
Bull_XM_V2,USDCAD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,115,114,229,50.218341,0.000891,-0.000917,0.102416,-0.104554,-0.002137,670000.0,26.8,-7569.32,-67054.43
Bear_XM_V2,USDCAD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,65,83,148,43.918919,0.001018,-0.001175,0.066172,-0.097511,-0.031339,670000.0,26.8,-24963.36,-92017.79
Bull_XM_V2,USDCAD,2025-03-13 02:00:00-04:00,2026-03-12 11:00:00-04:00,118,111,229,51.528384,0.000712,-0.000784,0.084049,-0.087071,-0.003023,670000.0,26.8,-8162.28,-100180.07
Bear_XM_V2,USDCAD,2025-03-13 02:00:00-04:00,2026-03-12 11:00:00-04:00,119,79,198,60.101010,0.001017,-0.001069,0.121066,-0.084470,0.036596,670000.0,26.8,19213.09,-80966.98


#### Trades (Last 10)

In [31]:
g[[*gains_cols, "SMA4_Slope", "Bull_XM_V2", "Bear_XM_V2"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,Bull_XM_V2,Bear_XM_V2
Date,,,,,,,,,,
2026-02-25 10:30:00-05:00,1.0,0.0,0.000992,-0.000992,0.000992,2026-02-25 10:45:00-05:00,2026-02-25 16:45:00-05:00,-64.668399,False,True
2026-02-25 10:45:00-05:00,1.0,0.0,0.001019,-0.001019,0.000580,2026-02-25 11:00:00-05:00,2026-02-25 16:45:00-05:00,-70.872390,False,True
2026-02-27 10:00:00-05:00,1.0,0.0,0.001650,-0.001650,0.001650,2026-02-27 10:15:00-05:00,2026-02-27 10:15:00-05:00,-72.431031,False,True
2026-02-27 10:45:00-05:00,1.0,0.0,0.001812,-0.001812,0.000260,2026-02-27 11:00:00-05:00,2026-02-27 16:45:00-05:00,-74.250425,False,True
2026-02-27 11:00:00-05:00,0.0,1.0,0.001578,-0.001578,-0.001578,2026-02-27 11:15:00-05:00,2026-02-27 16:00:00-05:00,-73.964018,False,True
2026-03-05 07:45:00-05:00,1.0,0.0,0.000787,-0.000787,0.000465,2026-03-05 08:00:00-05:00,2026-03-05 08:00:00-05:00,-54.055052,False,True
2026-03-05 08:15:00-05:00,0.0,1.0,0.000872,-0.000872,-0.000872,2026-03-05 08:30:00-05:00,2026-03-05 08:30:00-05:00,-66.194056,False,True
2026-03-09 06:30:00-04:00,1.0,0.0,0.000915,-0.000915,0.000915,2026-03-09 06:45:00-04:00,2026-03-09 08:00:00-04:00,-76.266179,False,True
2026-03-09 06:45:00-04:00,1.0,0.0,0.000694,-0.000694,0.000694,2026-03-09 07:00:00-04:00,2026-03-09 08:00:00-04:00,-70.820992,False,True


### Trend Continuation

In [32]:
long_signal = "Bull_TC"
short_signal = "Bear_TC"

stats, gains = range_based_multi_year_stats(cad_df_list, long_signal, short_signal, 1, 1.5, get_range_signal_gains, trading_session="new york")

#### Results (5 Years)

In [33]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_TC,USDCAD,2021-03-12 07:00:00-05:00,2022-03-11 16:00:00-05:00,110,117,227,48.458150,0.001114,-0.000986,0.122572,-0.115332,0.007240,670000.0,26.8,-1232.80,-1232.80
Bear_TC,USDCAD,2021-03-12 07:00:00-05:00,2022-03-11 16:00:00-05:00,105,132,237,44.303797,0.001262,-0.000898,0.132537,-0.118601,0.013936,670000.0,26.8,2985.27,1752.47
Bull_TC,USDCAD,2022-03-11 07:00:00-05:00,2023-03-10 16:00:00-05:00,109,148,257,42.412451,0.001702,-0.001257,0.185478,-0.185995,-0.000517,670000.0,26.8,-7233.91,-5481.44
Bear_TC,USDCAD,2022-03-11 07:00:00-05:00,2023-03-10 16:00:00-05:00,106,122,228,46.491228,0.001536,-0.001179,0.162765,-0.142611,0.020154,670000.0,26.8,7392.61,1911.17
Bull_TC,USDCAD,2023-03-14 07:00:00-04:00,2024-03-12 16:00:00-04:00,89,133,222,40.090090,0.001148,-0.000806,0.102149,-0.106409,-0.004260,670000.0,26.8,-8803.80,-6892.63
Bear_TC,USDCAD,2023-03-14 07:00:00-04:00,2024-03-12 16:00:00-04:00,110,112,222,49.549550,0.001042,-0.000873,0.114654,-0.097776,0.016878,670000.0,26.8,5358.33,-1534.30
Bull_TC,USDCAD,2024-03-13 07:00:00-04:00,2025-03-12 16:00:00-04:00,115,109,224,51.339286,0.000901,-0.000858,0.103559,-0.092644,0.010915,670000.0,26.8,1309.85,-224.45
Bear_TC,USDCAD,2024-03-13 07:00:00-04:00,2025-03-12 16:00:00-04:00,75,111,186,40.322581,0.000984,-0.000814,0.073810,-0.089492,-0.015682,670000.0,26.8,-15492.07,-15716.52
Bull_TC,USDCAD,2025-03-13 07:00:00-04:00,2026-03-12 16:00:00-04:00,96,113,209,45.933014,0.000885,-0.000809,0.084994,-0.091394,-0.006400,670000.0,26.8,-9889.20,-25605.72
Bear_TC,USDCAD,2025-03-13 07:00:00-04:00,2026-03-12 16:00:00-04:00,67,128,195,34.358974,0.001140,-0.000823,0.076391,-0.105312,-0.028921,670000.0,26.8,-24603.24,-50208.96


#### Trades (Last 10)

In [34]:
g[[*gains_cols, "SMA4_Slope", "Bull_TC", "Bear_TC"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,Bull_TC,Bear_TC
Date,,,,,,,,,,
2026-03-02 07:30:00-05:00,0.0,1.0,0.001316,-0.000878,-0.000878,2026-03-02 07:45:00-05:00,2026-03-02 07:45:00-05:00,-6.889837,False,True
2026-03-04 05:00:00-05:00,0.0,1.0,0.002006,-0.001337,-0.001337,2026-03-04 05:15:00-05:00,2026-03-04 05:15:00-05:00,-64.668399,False,True
2026-03-04 08:00:00-05:00,0.0,1.0,0.002443,-0.001629,-0.001629,2026-03-04 08:15:00-05:00,2026-03-04 08:45:00-05:00,59.036243,False,True
2026-03-09 03:15:00-04:00,0.0,1.0,0.001221,-0.000814,-0.000814,2026-03-09 03:30:00-04:00,2026-03-09 03:30:00-04:00,-22.006917,False,True
2026-03-10 04:30:00-04:00,0.0,1.0,0.001176,-0.000784,-0.000784,2026-03-10 04:45:00-04:00,2026-03-10 04:45:00-04:00,51.709837,False,True
2026-03-10 05:00:00-04:00,0.0,1.0,0.001222,-0.000815,-0.000815,2026-03-10 05:15:00-04:00,2026-03-10 05:45:00-04:00,27.135140,False,True
2026-03-10 10:00:00-04:00,1.0,0.0,0.001721,-0.001147,0.001721,2026-03-10 10:15:00-04:00,2026-03-10 14:15:00-04:00,71.588895,False,True
2026-03-10 10:30:00-04:00,0.0,1.0,0.001785,-0.001190,-0.001190,2026-03-10 10:45:00-04:00,2026-03-10 11:30:00-04:00,-67.238033,False,True
2026-03-11 02:00:00-04:00,0.0,1.0,0.000446,-0.000298,-0.000298,2026-03-11 02:15:00-04:00,2026-03-11 02:15:00-04:00,-4.763642,False,True
